|<h2>Course:</h2>|<h1><a href="https://derivingsystems.com/course.html" target="_blank">Build your own vLLM: inference engines from the memory system up</a></h1>|
|-|:-:|
|<h2>Part 0:</h2>|<h1>From a Program to a Model<h1>|
|<h2>Section:</h2>|<h1>The map<h1>|
|<h2>Lecture:</h2>|<h1><b>Where your instinct breaks: six predictions<b></h1>|

<br>

<h5><b>Course repo:</b> <a href="https://github.com/Venugopalan2610/vllm-from-scratch" target="_blank">github.com/Venugopalan2610/vllm-from-scratch</a></h5>
<h5><b>The derivations:</b> <a href="https://derivingsystems.com" target="_blank">derivingsystems.com</a></h5>
<i>The notebooks build the intuition. The ladder in app/ makes you build the thing.</i>

# Your instinct, measured

Your instinct as a software engineer has two parts. The **shape of your
search** (layers, narrowing, patterns) transfers to an engine completely. The
last notebook gave you the layers.

The **assumptions under your search** do not transfer. You learned them on
CPUs, networks and databases. This notebook takes six of them. For each one:

1. You **predict** with your old instinct. Write the number in the cell
   before you run the measurement. Do not skip this. A prediction that you
   write after the result teaches you nothing.
2. The next cell **measures** it on your GPU.
3. You see where the instinct fails, and why.

A wrong prediction is the point of this notebook, not a failure. When one of
yours is wrong, write it down with `./vc note "..."`.

In [1]:
# find the repo root. The directory you start from does not matter.
import sys
from pathlib import Path
ROOT = next(folder for folder in [Path.cwd(), *Path.cwd().parents]
            if (folder/'cudalib').is_dir())
sys.path.insert(0, str(ROOT))

import time
import numpy as np
import torch
import matplotlib.pyplot as plt
import cudalib

import matplotlib_inline.backend_inline
matplotlib_inline.backend_inline.set_matplotlib_formats('svg')

from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL = 'Qwen/Qwen3-1.7B'
tokenizer = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(MODEL, dtype=torch.bfloat16).cuda().eval()
WEIGHT_BYTES = sum(parameter.numel() * parameter.element_size() for parameter in model.parameters())

# Two numbers of YOUR card. Part 0, section 3, measures them in detail.
PEAK_BANDWIDTH = cudalib.peak_bandwidth() * 1e9     # bytes/s
PEAK_FLOPS = cudalib.matmul_flops()                  # FLOP/s, bf16
print(f'weights:   {WEIGHT_BYTES/1e9:.2f} GB')
print(f'bandwidth: {PEAK_BANDWIDTH/1e9:.0f} GB/s')
print(f'compute:   {PEAK_FLOPS/1e12:.1f} TFLOP/s')

PREDICTED, MEASURED = {}, {}

/home/venugopalan/vllm-from-scratch/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 311/311 [00:00<00:00, 23222.46it/s]

weights:   3.44 GB
bandwidth: 295 GB/s
compute:   67.1 TFLOP/s


# 1. "Each request costs its own share"

In a web service, 32 requests cost about 32 times the work of one request.

One **decode** step makes one new token for each request in the batch.
Predict: how long is one step for 32 requests, divided by one step for 1
request?

In [2]:
# PREDICT, before you run the next cell.
PREDICTED['1. 32 requests / 1 request'] = None      # your number

In [3]:
@torch.inference_mode()
def step_ms(model, batch_size):
    """One decode step: one new token for each of batch_size requests."""
    tokens = torch.randint(0, 1000, (batch_size, 1), device='cuda')
    return cudalib.bench_ms(lambda: model(tokens, use_cache=False), iters=20, warmup=5, best_of=3)

batch_sizes = [1, 2, 4, 8, 16, 32]
step_times = [step_ms(model, batch_size) for batch_size in batch_sizes]
for batch_size, milliseconds in zip(batch_sizes, step_times):
    print(f'{batch_size:>3} requests: {milliseconds/step_times[0]:5.2f}x the time of 1 request')
MEASURED['1. 32 requests / 1 request'] = step_times[-1] / step_times[0]

  1 requests:  1.00x the time of 1 request
  2 requests:  1.04x the time of 1 request
  4 requests:  1.05x the time of 1 request
  8 requests:  1.06x the time of 1 request
 16 requests:  1.08x the time of 1 request
 32 requests:  1.15x the time of 1 request


The requests do not each pay for their own step. The model reads all its
weights one time for each step, and every request in the batch uses that one
read. So the 32nd request is almost free.

The other side of the same fact: all the requests move together, in the same
step. If one request makes the step slow, every other request waits. Stage 11
is about that.

# 2. "Computation is the expensive part"

In a web service, you make code faster when it does less work.

At batch 1, one step does two things. It **reads** every weight from memory,
and it does the **arithmetic**: one multiply and one add for each weight.
Predict: what percentage of the step time is the arithmetic?

In [4]:
PREDICTED['2. arithmetic, % of the step'] = None      # your number, 0 to 100

In [5]:
read_ms = WEIGHT_BYTES / PEAK_BANDWIDTH * 1e3
parameters = WEIGHT_BYTES / 2                          # 2 bytes for each bf16 weight
arithmetic_ms = 2 * parameters / PEAK_FLOPS * 1e3
print(f'the read of the weights, at full bandwidth: {read_ms:7.2f} ms')
print(f'the arithmetic, at full speed:              {arithmetic_ms:7.2f} ms')
print(f'the measured step at batch 1:               {step_times[0]:7.2f} ms')
MEASURED['2. arithmetic, % of the step'] = 100 * arithmetic_ms / step_times[0]
print(f'-> the arithmetic is {MEASURED["2. arithmetic, % of the step"]:.1f}% of the step')

the read of the weights, at full bandwidth:   11.67 ms
the arithmetic, at full speed:                 0.05 ms
the measured step at batch 1:                 11.63 ms
-> the arithmetic is 0.4% of the step


The step waits for bytes. The arithmetic units have almost nothing to do. So
"do less work" does not help here. Two things help: read fewer bytes (Part 7
makes the weights smaller), or serve more requests with each read (the batch,
from assumption 1).

Code that waits for memory, not for arithmetic, is **memory-bound**. It is the
GPU version of "I/O-bound". Part 1 derives the exact point where this changes.

# 3. "The state of one request is small"

In a web service, the state of one request is a few kilobytes. The big state
is in the database.

For each token of a conversation, the model keeps a memo of its work, so that
it never does that work again. That memo is the **KV cache**. Its size for one
token comes from the model config: the schema.

Predict: how many tokens long must one conversation be before its memo is as
large as the whole model?

In [6]:
PREDICTED['3. tokens until the memo = the model'] = None      # your number

In [7]:
config = model.config
# K and V, for each layer, for each KV head, head_dim numbers of 2 bytes
kv_bytes_per_token = 2 * config.num_hidden_layers * config.num_key_value_heads * config.head_dim * 2
print(f'memo for one token: {kv_bytes_per_token/1024:.0f} KB   (from the config)')

# Check the formula against the real memo of a 4096-token prompt.
with torch.inference_mode():
    tokens = torch.randint(0, 1000, (1, 4096), device='cuda')
    cache = model(tokens, use_cache=True).past_key_values
    real_bytes = sum(layer.keys.numel() * layer.keys.element_size()
                     + layer.values.numel() * layer.values.element_size()
                     for layer in cache.layers)
print(f'memo for 4096 tokens: {real_bytes/1e6:.0f} MB measured, '
      f'{kv_bytes_per_token*4096/1e6:.0f} MB from the formula')
del cache, tokens
torch.cuda.empty_cache()

MEASURED['3. tokens until the memo = the model'] = WEIGHT_BYTES / kv_bytes_per_token
print(f'-> at {MEASURED["3. tokens until the memo = the model"]:,.0f} tokens, '
      'one conversation needs as much memory as the whole model')

memo for one token: 112 KB   (from the config)
memo for 4096 tokens: 470 MB measured, 470 MB from the formula


-> at 30,004 tokens, one conversation needs as much memory as the whole model


So on a GPU, the memory, not the CPU, sets how many users fit at the same
time. Every request carries a large state, and it grows by one token at each
step. Stage 06 manages that memory in pages, the way an operating system
manages the memory of processes.

# 4. "A bug raises an error"

In a web service, most bugs raise an exception, or return a 500.

Here is a real bug from a real engine: it gives a new request the memo table
of the request before it. Predict: what does the user see? Write `'error'`,
`'garbage'` (text that makes no sense) or `'fluent'` (normal text).

In [8]:
PREDICTED['4. what the user sees'] = None      # 'error', 'garbage' or 'fluent'

In [9]:
@torch.inference_mode()
def generate(prompt, num_tokens=40, old_cache=None):
    """Greedy generation with an explicit memo table. With `old_cache`, it
    reuses the memo of an earlier request: the bug."""
    prompt_ids = tokenizer(prompt, return_tensors='pt').input_ids.cuda()
    start = old_cache.get_seq_length() if old_cache is not None else 0
    forward_pass = model(prompt_ids, past_key_values=old_cache, use_cache=True)
    cache, token, tokens = forward_pass.past_key_values, None, []
    for index in range(num_tokens):
        token = forward_pass.logits[0, -1].argmax().item()
        tokens.append(token)
        position = torch.tensor([[start + prompt_ids.shape[1] + index]], device='cuda')
        forward_pass = model(torch.tensor([[token]], device='cuda'), position_ids=position,
                       past_key_values=cache, use_cache=True)
        cache = forward_pass.past_key_values
    return tokens, cache

PROMPT = 'The three most important ideas in operating systems are'
right, _ = generate(PROMPT)
_, earlier_cache = generate('My favourite recipe for a chocolate cake needs')
wrong, _ = generate(PROMPT, old_cache=earlier_cache)          # no exception here

print('correct:  ', repr(tokenizer.decode(right)))
print('with bug: ', repr(tokenizer.decode(wrong)))
different = sum(right_token != wrong_token for right_token, wrong_token in zip(right, wrong))
print(f'\nexceptions: 0.   tokens that differ from the oracle: {different} of {len(right)}')
MEASURED['4. what the user sees'] = 'fluent'

correct:   ': 1. Process management, 2. Memory management, and 3. File systems. 1. Process management: The operating system manages the execution of programs. It is responsible for creating'
with bug:  ': 1. 2. 3.  The user is asking for a chocolate cake recipe, but also wants to know the three most important ideas in operating systems.  How can I address'

exceptions: 0.   tokens that differ from the oracle: 33 of 40


No error, and the text is fluent. Nothing in a log tells you that it is
wrong. Only a comparison with a version that you trust finds it.
That trusted version is an **oracle**. Stage 02 has a check for exactly this
bug, and the course keeps an oracle for every fast part that you build.

# 5. "The framework overhead is small"

In a web service, 1 ms of Python is nothing next to a 50 ms database call.

One decode step is about a thousand small GPU functions (**kernels**). Python
starts each one with a call, a **kernel launch**, and each call has a fixed
cost. A **CUDA graph** records all the launches of one step one time, and
then replays them with no Python at all. It is like a prepared statement.

Take a smaller model, Qwen3-0.6B, so the GPU has less work in each step.
Predict: what percentage of its step is Python and launches?

In [10]:
PREDICTED['5. Python and launches, % of the step'] = None      # your number, 0 to 100

In [11]:
@torch.inference_mode()
def eager_and_graph_ms(model):
    tokens = torch.randint(0, 1000, (1, 1), device='cuda')
    eager = cudalib.bench_ms(lambda: model(tokens, use_cache=False), iters=20, warmup=5, best_of=3)
    side = torch.cuda.Stream()
    side.wait_stream(torch.cuda.current_stream())
    with torch.cuda.stream(side):                     # warm up before the capture
        for _ in range(3):
            model(tokens, use_cache=False)
    torch.cuda.current_stream().wait_stream(side)
    graph = torch.cuda.CUDAGraph()
    with torch.cuda.graph(graph):
        model(tokens, use_cache=False)
    replay = cudalib.bench_ms(graph.replay, iters=20, warmup=5, best_of=3)
    del graph
    return eager, replay

small = AutoModelForCausalLM.from_pretrained('Qwen/Qwen3-0.6B', dtype=torch.bfloat16).cuda().eval()
for name, some_model in [('Qwen3-0.6B', small), ('Qwen3-1.7B', model)]:
    eager, replay = eager_and_graph_ms(some_model)
    overhead = 100 * (eager - replay) / eager
    print(f'{name}: step {eager:6.2f} ms, the same step replayed {replay:6.2f} ms '
          f'-> Python and launches are {overhead:4.1f}% of the step')
    if name == 'Qwen3-0.6B':
        MEASURED['5. Python and launches, % of the step'] = overhead
del small
torch.cuda.empty_cache()

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 311/311 [00:00<00:00, 17307.23it/s]

Qwen3-0.6B: step   9.24 ms, the same step replayed   5.05 ms -> Python and launches are 45.4% of the step


Qwen3-1.7B: step  11.64 ms, the same step replayed  10.56 ms -> Python and launches are  9.3% of the step


On the small model, Python and the launches are a large part of the step.
On the larger model, they almost disappear. Why?

The launches are **asynchronous**. Python puts each kernel in a queue and
continues at once, while the GPU works on the queue. So a step costs about
**the larger of** the Python time and the GPU time, not the sum. When the
GPU work is long, it hides the Python. When the GPU work is short, the Python
is the step.

Stage 12 builds the graph replay. Part 0, section 4, shows you how this same
queue can make a stopwatch lie to you.

# 6. "The same input gives the same output"

In a web service, a pure function gives the same result for the same input.

**Greedy decoding** always takes the top choice, so it has no randomness.
Run it on one prompt two times: alone, and in a batch of 2 copies of the
same prompt. Predict: how many of the 64 tokens differ?

In [12]:
PREDICTED['6. tokens that differ, of 64'] = None      # your number

In [13]:
@torch.inference_mode()
def greedy(batch_ids, num_tokens=64):
    """Greedy tokens of row 0, and the top-2 scores at each position."""
    forward_pass = model(batch_ids, use_cache=True)
    cache, tokens, top2 = forward_pass.past_key_values, [], []
    for _ in range(num_tokens):
        scores = forward_pass.logits[:, -1].float()
        top2.append(scores[0].topk(2).values.tolist())
        next_tokens = scores.argmax(-1, keepdim=True)
        tokens.append(next_tokens[0].item())
        forward_pass = model(next_tokens, past_key_values=cache, use_cache=True)
        cache = forward_pass.past_key_values
    return tokens, top2

def differ(prompt):
    prompt_ids = tokenizer(prompt, return_tensors='pt').input_ids.cuda()
    alone, top2 = greedy(prompt_ids)
    in_batch, _ = greedy(prompt_ids.repeat(2, 1))
    first = next((i for i, (alone_token, batch_token) in enumerate(zip(alone, in_batch)) if alone_token != batch_token), None)
    return sum(alone_token != batch_token for alone_token, batch_token in zip(alone, in_batch)), first, top2

num_different, first, top2 = differ(PROMPT)
MEASURED['6. tokens that differ, of 64'] = num_different
print(f'{num_different} of 64 tokens differ.')
if first is not None:
    print(f'The first difference is at token {first}. There, the top two scores '
          f'at batch 1 were {top2[first][0]:.3f} and {top2[first][1]:.3f}.')

# One prompt is one sample. Try more.
PROMPTS = ['The capital of France is', 'def fibonacci(n):', 'Water boils at',
           'The best way to learn a language is', 'In 1969, the first person to walk on the moon',
           'A good unit test should', 'The difference between a list and a tuple in Python is']
counts = [differ(prompt)[0] for prompt in PROMPTS]
print(f'\nOther prompts: {sum(num_changed > 0 for num_changed in counts)} of {len(PROMPTS)} changed somewhere '
      f'in 64 tokens. Tokens that differ: {counts}')

63 of 64 tokens differ.
The first difference is at token 1. There, the top two scores at batch 1 were 21.625 and 21.625.



Other prompts: 4 of 7 changed somewhere in 64 tokens. Tokens that differ: [0, 0, 9, 0, 45, 2, 52]


The math of one row does not depend on the other rows. But the GPU adds the
numbers of a matrix product in a different order when the batch size
changes. Floating-point addition is not associative: `(a + b) + c` and
`a + (b + c)` can round differently. The difference is tiny. At a position
where the top two choices are almost equal, it is enough to flip the choice,
and after that the two texts are different conversations.

Users report this as "the same prompt gives a different answer when the
server is busy". `LORE.md`, section 12, measures how often it happens.

# Your predictions, against the measurements

In [14]:
print(f"{'assumption':<42} {'you predicted':>14} {'measured':>12}")
for key in MEASURED:
    predicted, measured = PREDICTED.get(key), MEASURED[key]
    show = lambda value: f'{value:.2f}' if isinstance(value, float) else str(value)
    print(f'{key:<42} {show(predicted):>14} {show(measured):>12}')

assumption                                  you predicted     measured
1. 32 requests / 1 request                           None         1.15
2. arithmetic, % of the step                         None         0.44
3. tokens until the memo = the model                 None     30004.45
4. what the user sees                                None       fluent
5. Python and launches, % of the step                None        45.35
6. tokens that differ, of 64                         None           63


Which predictions were far from the measurement? For each one, write one line:

```bash
./vc note "predicted 32x for 32 requests, measured about 1x: the weights are read one time for the whole batch"
```

### What to remember from this notebook

| Your instinct says | On a GPU engine |
|---|---|
| Each request costs its own share | All the requests share one step, and one read of the weights. |
| Computation is the expensive part | The step waits for bytes. The arithmetic is a small part. |
| The state of one request is small | The memo of one long conversation can be larger than the model. |
| A bug raises an error | A bug gives fluent, wrong text. Keep an oracle. |
| The framework overhead is small | Python and launches can be a large part of a step. |
| The same input gives the same output | A different batch size can change the text. |

`docs/METHOD.md` has this table, and the six moves that the rest of the
course uses to find more assumptions like these.